In [3]:
# Setup
import os, re, json, gzip, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_PER_CATEGORY = 2000

random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/NLP_Assignment3_Transformers_RAG"
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

files_map = {
    "sports": "/content/sports.json.gz",
    "cellphones": "/content/cellphones.json.gz",
    "home": "/content/home.json.gz"
}

# Clean
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Sentiment
def rating_to_sentiment(rating):
    if rating in [1, 2]:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

# Loader
def load_json_gz(path, category, sample_size):
    rows = []

    if not os.path.exists(path):
        print("Missing file:", path)
        return pd.DataFrame()

    with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            try:
                item = json.loads(line)

                review = item.get("reviewText", "")
                rating = item.get("overall", None)

                if review and rating is not None:
                    rows.append({
                        "review_text": review,
                        "overall": int(float(rating)),
                        "category": category
                    })

                if len(rows) >= sample_size * 3:
                    break

            except:
                pass

    df = pd.DataFrame(rows)
    df = df.dropna()
    df = df.drop_duplicates(subset=["review_text"])

    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=SEED)

    return df.reset_index(drop=True)

# Load
all_dfs = []

for category, path in files_map.items():
    print("Loading:", category)
    cat_df = load_json_gz(path, category, SAMPLE_PER_CATEGORY)
    print(category, cat_df.shape)
    all_dfs.append(cat_df)

df = pd.concat(all_dfs, ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Process
df["review_text"] = df["review_text"].apply(clean_text)
df = df[df["review_text"].str.len() > 10].reset_index(drop=True)

sentiment_to_id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["sentiment"] = df["overall"].apply(rating_to_sentiment)
df["sentiment_label"] = df["sentiment"].map(sentiment_to_id)

# Length
df["token_count"] = df["review_text"].apply(lambda x: len(x.split()))

q1 = df["token_count"].quantile(0.33)
q2 = df["token_count"].quantile(0.66)

def length_class(count):
    if count <= q1:
        return "short"
    elif count <= q2:
        return "medium"
    else:
        return "long"

length_to_id = {
    "short": 0,
    "medium": 1,
    "long": 2
}

df["review_length_class"] = df["token_count"].apply(length_class)
df["length_label"] = df["review_length_class"].map(length_to_id)

# Columns
final_df = df[
    [
        "review_text",
        "overall",
        "sentiment",
        "sentiment_label",
        "review_length_class",
        "length_label",
        "token_count",
        "category"
    ]
].copy()

# Split
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=SEED,
    stratify=final_df["sentiment_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["sentiment_label"]
)

# Save
processed_path = os.path.join(DATA_DIR, "processed_reviews.csv")
train_path = os.path.join(DATA_DIR, "train.csv")
val_path = os.path.join(DATA_DIR, "val.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

final_df.to_csv(processed_path, index=False)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

# Summary
summary = {
    "phase": "Phase 1",
    "sample_per_category": SAMPLE_PER_CATEGORY,
    "total_samples": int(len(final_df)),
    "categories": list(files_map.keys()),
    "train_samples": int(len(train_df)),
    "val_samples": int(len(val_df)),
    "test_samples": int(len(test_df)),
    "sentiment_mapping": {
        "1-2": "negative",
        "3": "neutral",
        "4-5": "positive"
    },
    "derived_feature": "review_length_class",
    "length_classes": length_to_id,
    "short_max_tokens": float(q1),
    "medium_max_tokens": float(q2)
}

summary_path = os.path.join(RESULTS_DIR, "phase1_summary.json")

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

# Output
print("\nPHASE 1 COMPLETE")
print("Total:", final_df.shape)
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nCategory:")
print(final_df["category"].value_counts())

print("\nSentiment:")
print(final_df["sentiment"].value_counts())

print("\nLength:")
print(final_df["review_length_class"].value_counts())

print("\nSaved:")
print(processed_path)
print(train_path)
print(val_path)
print(test_path)
print(summary_path)

Loading: sports
sports (2000, 3)
Loading: cellphones
cellphones (2000, 3)
Loading: home
home (2000, 3)

PHASE 1 COMPLETE
Total: (5998, 8)
Train: (4198, 8)
Val: (900, 8)
Test: (900, 8)

Category:
category
sports        2000
cellphones    1999
home          1999
Name: count, dtype: int64

Sentiment:
sentiment
positive    4986
negative     578
neutral      434
Name: count, dtype: int64

Length:
review_length_class
long      2039
short     2032
medium    1927
Name: count, dtype: int64

Saved:
/content/NLP_Assignment3_Transformers_RAG/data/processed_reviews.csv
/content/NLP_Assignment3_Transformers_RAG/data/train.csv
/content/NLP_Assignment3_Transformers_RAG/data/val.csv
/content/NLP_Assignment3_Transformers_RAG/data/test.csv
/content/NLP_Assignment3_Transformers_RAG/results/phase1_summary.json


In [4]:
# Download
from google.colab import files

files.download("/content/NLP_Assignment3_Transformers_RAG/data/processed_reviews.csv")
files.download("/content/NLP_Assignment3_Transformers_RAG/data/train.csv")
files.download("/content/NLP_Assignment3_Transformers_RAG/data/val.csv")
files.download("/content/NLP_Assignment3_Transformers_RAG/data/test.csv")
files.download("/content/NLP_Assignment3_Transformers_RAG/results/phase1_summary.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# Setup
import os, json, re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Config
SEED = 42
MAX_LEN = 128
MIN_FREQ = 2
MAX_VOCAB_SIZE = 20000
BATCH_SIZE = 32

torch.manual_seed(SEED)

# Paths
BASE_DIR = "/content/NLP_Assignment3_Transformers_RAG"
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

train_path = os.path.join(DATA_DIR, "train.csv")
val_path = os.path.join(DATA_DIR, "val.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

# Fallback
if not os.path.exists(train_path):
    train_path = "/content/train.csv"
    val_path = "/content/val.csv"
    test_path = "/content/test.csv"

# Load CSV
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Loaded:")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# Tokenizer
def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-z0-9']+|[.,!?]", text)
    return tokens

# Word count
counter = Counter()

for text in train_df["review_text"]:
    counter.update(tokenize(text))

# Special tokens
vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}

# Build vocab
for word, freq in counter.most_common():
    if freq < MIN_FREQ:
        continue

    if len(vocab) >= MAX_VOCAB_SIZE:
        break

    if word not in vocab:
        vocab[word] = len(vocab)

# ID maps
id_to_word = {idx: word for word, idx in vocab.items()}

pad_id = vocab["<PAD>"]
unk_id = vocab["<UNK>"]
bos_id = vocab["<BOS>"]
eos_id = vocab["<EOS>"]

print("Vocab size:", len(vocab))

# Encode text
def encode_text(text, max_len=MAX_LEN):
    tokens = tokenize(text)

    ids = [bos_id]

    for token in tokens:
        ids.append(vocab.get(token, unk_id))

    ids.append(eos_id)

    # Truncate
    if len(ids) > max_len:
        ids = ids[:max_len]
        ids[-1] = eos_id

    # Mask
    attention_mask = [1] * len(ids)

    # Padding
    while len(ids) < max_len:
        ids.append(pad_id)
        attention_mask.append(0)

    return ids, attention_mask

# Dataset class
class ReviewDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["review_text"].tolist()
        self.sentiment_labels = dataframe["sentiment_label"].tolist()
        self.length_labels = dataframe["length_label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_text(self.texts[idx])

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "sentiment_label": torch.tensor(self.sentiment_labels[idx], dtype=torch.long),
            "length_label": torch.tensor(self.length_labels[idx], dtype=torch.long)
        }

# Datasets
train_dataset = ReviewDataset(train_df)
val_dataset = ReviewDataset(val_df)
test_dataset = ReviewDataset(test_df)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Batch test
batch = next(iter(train_loader))

print("\nBatch check:")
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("sentiment_label:", batch["sentiment_label"].shape)
print("length_label:", batch["length_label"].shape)

# Save folder
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save vocab
vocab_path = os.path.join(RESULTS_DIR, "vocab.json")

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=4)

# Summary
phase2_summary = {
    "phase": "Phase 2",
    "max_len": MAX_LEN,
    "min_freq": MIN_FREQ,
    "max_vocab_size": MAX_VOCAB_SIZE,
    "final_vocab_size": len(vocab),
    "batch_size": BATCH_SIZE,
    "train_batches": len(train_loader),
    "val_batches": len(val_loader),
    "test_batches": len(test_loader),
    "special_tokens": {
        "PAD": pad_id,
        "UNK": unk_id,
        "BOS": bos_id,
        "EOS": eos_id
    }
}

# Save summary
summary_path = os.path.join(RESULTS_DIR, "phase2_summary.json")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(phase2_summary, f, indent=4)

# Done
print("\nPHASE 2 COMPLETE")
print("Saved vocab:", vocab_path)
print("Saved summary:", summary_path)

Loaded:
Train: (4198, 8)
Val: (900, 8)
Test: (900, 8)
Vocab size: 8122

Batch check:
input_ids: torch.Size([32, 128])
attention_mask: torch.Size([32, 128])
sentiment_label: torch.Size([32])
length_label: torch.Size([32])

PHASE 2 COMPLETE
Saved vocab: /content/NLP_Assignment3_Transformers_RAG/results/vocab.json
Saved summary: /content/NLP_Assignment3_Transformers_RAG/results/phase2_summary.json


In [6]:
# Download
from google.colab import files

files.download("/content/NLP_Assignment3_Transformers_RAG/results/vocab.json")
files.download("/content/NLP_Assignment3_Transformers_RAG/results/phase2_summary.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>